# PlantCLEF 2026 — soumission

Notebook de soumission pour Kaggle. La structure est celle de `submission-method.md` : les cellules 1 à 4, 6 et 7 sont fixes. Seule `predict_species` (cellule 5) change d'un run à l'autre.

**Ce run** repère les plantes par la couleur (bleu, violet, rose, rouge, orange, jaune saturé, et blanc franc), place un carré centré sur chaque amas, puis passe ces carrés dans le modèle. Le score d'une espèce est le maximum de ses probabilités softmax sur les carrés. La fonction renvoie au plus 10 espèces de score au moins `0.1`, par score décroissant. Aucune plante détectée, ou aucune espèce au-dessus du seuil, donne une liste vide.


In [ ]:
import os, csv
import numpy as np
import pandas as pd
import torch
import timm
from PIL import Image
from tqdm import tqdm


In [ ]:
C = "/kaggle/input/competitions/plantclef-2026"
TEST_CSV    = f"{C}/PlantCLEF2025_test.csv"
SPECIES_CSV = f"{C}/species_ids.csv"
IMG_DIR     = f"{C}/PlantCLEF2025_test_images/PlantCLEF2025_test_images"
MODEL_CKPT  = "/kaggle/input/models/juliostat/dinov2_patch14_reg4_onlyclassifier_then_all/pytorch/default/3/model_best.pth.tar"

ID_COL  = "quadrat_id"   # vérifié : colonnes du CSV de test = quadrat_id, author, date, license
IMG_EXT = ".jpg"         # vérifié : les 2 105 images sont en .jpg, noms = quadrat_id + extension

def image_path(quadrat_id: str) -> str:
    return f"{IMG_DIR}/{quadrat_id}{IMG_EXT}"

print("GPU :", torch.cuda.is_available())


In [ ]:
df_species = pd.read_csv(SPECIES_CSV)
class_map = df_species["species_id"].to_dict()   # indice de sortie -> species_id

# Contrôle : d'après la page Zenodo des modèles, la sortie 2 correspond à l'espèce 1355870.
assert class_map[2] == 1355870, "species_ids.csv ne suit pas l'ordre des sorties du modèle (class_mapping.txt n'est pas dans les entrées)"


In [ ]:
device = torch.device("cuda")
model = timm.create_model(
    "vit_base_patch14_reg4_dinov2.lvd142m",
    pretrained=False,                 # pas de téléchargement : les poids viennent de MODEL_CKPT
    num_classes=len(df_species),
    checkpoint_path=MODEL_CKPT,
)
model = model.to(device).eval()
data_config = timm.data.resolve_model_data_config(model)


In [ ]:
from matplotlib.colors import rgb_to_hsv
from scipy import ndimage as ndi

# Carrés centrés sur les plantes : masque de couleur, un carré par amas, max des softmax.
SCORE_MIN = 0.1    # probabilité softmax minimale pour retenir une espèce
TOP_K = 10         # nombre maximal d'espèces renvoyées
CROP_BATCH = 16

WORK_SIDE = 800
KEEP_WHITE = True  # False si des reflets ou un cadre blanc sont pris pour des plantes
MERGE_DIST = 0.01  # distance de fusion, en fraction du côté : plus grande, les fleurs voisines n'en font qu'une
MIN_AREA = 0.0003  # taille minimale d'un amas, en fraction de l'image réduite
MARGIN = 1.6       # part de contexte autour de la plante
MIN_SIDE = 0.15    # taille du carré, en fraction du petit côté (évite d'agrandir une fleur minuscule jusqu'au flou)
MAX_SIDE = 0.5
IOU_MAX = 0.4      # recouvrement maximal entre deux carrés
MAX_CROPS = 30

_RESAMPLE = {
    "nearest": Image.Resampling.NEAREST,
    "bilinear": Image.Resampling.BILINEAR,
    "bicubic": Image.Resampling.BICUBIC,
}


def color_mask(rgb, keep_white=True):
    """rgb : (h, w, 3) uint8. Masque des pixels ni verts, ni marrons, ni gris."""
    hsv = rgb_to_hsv(rgb / 255.0)
    h, s, v = hsv[..., 0] * 360, hsv[..., 1], hsv[..., 2]
    mask = (((h >= 170) & (h < 345) & (s >= 0.25) & (v >= 0.25))       # bleu, violet, rose
            | (((h >= 345) | (h < 50)) & (s >= 0.35) & (v >= 0.75))     # rouge, orange vif
            | ((h >= 50) & (h < 70) & (s >= 0.45) & (v >= 0.6)))        # jaune saturé
    if keep_white:
        mask |= (s < 0.15) & (v >= 0.9)                                 # blanc franc
    return mask


def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter)


def detect_plants(img, work_side=WORK_SIDE, keep_white=KEEP_WHITE, merge_dist=MERGE_DIST,
                  min_area=MIN_AREA, margin=MARGIN, min_side=MIN_SIDE, max_side=MAX_SIDE,
                  max_crops=MAX_CROPS, iou_max=IOU_MAX):
    """Carrés (x0, y0, x1, y1) en pixels de l'image d'origine, centrés sur chaque amas coloré."""
    width, height = img.size
    scale = work_side / max(width, height)
    small = img.resize((round(width * scale), round(height * scale)), Image.BILINEAR)
    mask = color_mask(np.asarray(small), keep_white)
    mask = ndi.binary_opening(mask, iterations=1)
    dilation = max(1, round(merge_dist * max(mask.shape)))
    blobs, _count = ndi.label(ndi.binary_dilation(mask, iterations=dilation))
    short_side = min(width, height)

    candidates = []
    for index, sl in enumerate(ndi.find_objects(blobs), start=1):
        area = int(mask[sl][blobs[sl] == index].sum())
        if area < min_area * mask.size:
            continue
        y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
        cx, cy = (x0 + x1) / 2 / scale, (y0 + y1) / 2 / scale
        side = np.clip(max(x1 - x0, y1 - y0) / scale * margin, min_side * short_side, max_side * short_side)
        bx = float(np.clip(cx - side / 2, 0, width - side))
        by = float(np.clip(cy - side / 2, 0, height - side))
        candidates.append((area, (round(bx), round(by), round(bx + side), round(by + side))))

    boxes = []
    for _area, box in sorted(candidates, key=lambda item: -item[0]):
        if all(iou(box, kept) < iou_max for kept in boxes):
            boxes.append(box)
        if len(boxes) >= max_crops:
            break
    return boxes


def crop_scores(img, boxes):
    """Softmax de chaque carré, de forme (n_carrés, n_classes)."""
    _channels, out_h, out_w = data_config["input_size"]
    resample = _RESAMPLE[data_config["interpolation"]]
    mean = np.asarray(data_config["mean"], dtype=np.float32)
    std = np.asarray(data_config["std"], dtype=np.float32)
    crops = []
    for box in boxes:
        crop = np.asarray(img.crop(box).resize((out_w, out_h), resample), dtype=np.float32) / 255.0
        crops.append((crop - mean) / std)
    batch = np.stack(crops)
    tensor = torch.from_numpy(np.ascontiguousarray(batch)).permute(0, 3, 1, 2)

    parts = []
    for start in range(0, len(tensor), CROP_BATCH):
        chunk = tensor[start:start + CROP_BATCH].to(device)
        with torch.no_grad(), torch.autocast("cuda"):
            parts.append(torch.softmax(model(chunk).float(), dim=1).cpu())
    return torch.cat(parts).numpy()


def species_from_scores(best) -> list[int]:
    """Espèces de score >= SCORE_MIN, au plus TOP_K, par score décroissant."""
    order = np.argsort(best)[::-1]
    species = []
    seen = set()
    for index in order:
        if best[index] < SCORE_MIN or len(species) >= TOP_K:
            break
        species_id = int(class_map[int(index)])
        if species_id not in seen:
            seen.add(species_id)
            species.append(species_id)
    return species


def predict_species(quadrat_id: str) -> list[int]:
    """Renvoie les espèces retenues pour un quadrat du test. Le modèle est déjà chargé."""
    path = image_path(quadrat_id)
    if not os.path.isfile(path):
        raise FileNotFoundError(path)

    image = Image.open(path).convert("RGB")
    boxes = detect_plants(image)
    if not boxes:
        return []
    return species_from_scores(crop_scores(image, boxes).max(axis=0))


In [ ]:
def create_submission_file(out_path: str = "/kaggle/working/submission.csv", predict=None) -> pd.DataFrame:
    """Parcourt PlantCLEF2025_test.csv dans l'ordre, appelle predict_species, écrit le CSV."""
    if predict is None:
        predict = predict_species

    ids = pd.read_csv(TEST_CSV, sep=";")[ID_COL].astype(str).tolist()   # ce CSV est séparé par des ;

    rows = []
    for qid in tqdm(ids):
        species = list(dict.fromkeys(int(s) for s in predict(qid)))   # int Python + sans doublons
        rows.append(str(species))

    df_sub = pd.DataFrame({"quadrat_id": ids, "species_ids": rows})
    df_sub.to_csv(out_path, index=False, quoting=csv.QUOTE_ALL)
    return df_sub


In [ ]:
df_sub = create_submission_file()
print(df_sub.shape)
print(df_sub.head())
print("quadrats sans espèce :", (df_sub["species_ids"] == "[]").sum())
